# 从零开始：机器何以学习？

## 介绍


🍋 **从一杯柠檬水开始的机器学习之旅**

在机器学习的世界中，最复杂的算法都建立在简单而直观的原理之上。本教程通过一个生动有趣的柠檬水模型，带领学习者从零开始构建对机器学习核心概念的理解。

🎯 **教程特色**

- 直观易懂：以“柠檬片数量预测放糖量”的日常场景作为切入点，将抽象的机器学习概念具象化
- 手动实现：不依赖高级框架，完全手写代码实现模型的前向传播、损失计算、反向传播和参数更新
- 循序渐进：从简单的固定步长更新到引入学习率的动态优化，逐步深化对训练机制的理解

📚 **学习收获**

- 理论基础：掌握模型训练的完整流程和优化原理
- 实践技能：学会实现基本的回归模型和训练循环
- 工程思维：了解模型保存、加载和推理的完整应用流程
- 调优技巧：理解学习率对模型收敛的重要影响

通过这个简单而完整的案例，希望能够帮你建立起对机器学习工作原理的直观认识，为后续深入学习复杂的神经网络模型奠定坚实基础。

## 任务背景

你是一个程序员，名字叫小吾。最近你注意到，城市街头每逢夏天，柠檬水摊位总是排起长队。一个想法在你脑中闪过：如果能训练一个机器人自动制作柠檬水，不就能开一家24小时营业的无人摊位，让钱源源不断地流进口袋了吗？

然而问题来了——机器人并不知道如何制作美味的柠檬水，尤其是关键的“柠檬片与糖”的配比。你虽然会编程，但对烹饪一窍不通，更别说柠檬水这种看似简单实则讲究平衡的艺术了。

幸运的是，你的好朋友小美是城市里最受欢迎的“夏日柠檬水”摊位老板，她的秘方据说能让路人喝完一杯还想再来一杯。经过一顿丰盛的晚餐和诚恳的请求，小美终于同意担任“柠檬水制作专家”，帮你训练这个机器人。



## 机器何以学习？

那么，问题来了，我们如何让机器人从小美那里学习到制作柠檬水的秘诀呢？

聪明的你很快找到一个方法：
1. 先请小美做一个完美的示范。
2. 让机器人先随便猜一个柠檬片对应的糖权值。
3. 然后让机器人试着做一杯柠檬水。
4. 接着和小美做的柠檬水进行比较。
5. 如果太甜了就减少一点糖，如果太酸了就加多一点糖。
6. 重复上面的步骤，最终我们就能学到小美制作柠檬水的秘诀了。

你已经掌握了机器学习的基本原理，现在开始你的机器学习之旅吧！

## 环境版本

打印环境的版本号，避免因为环境不同而导致程序不能复现。

In [11]:
from course_01.util import show_version

show_version()

+------+----------------------------------------------------------------------------------------+
| Info |                                   《动手学人工智能》                                   |
+------+----------------------------------------------------------------------------------------+
| 作者 |                                       吾辈亦有感                                       |
| 哔站 |                      https://space.bilibili.com/3461565265217538                       |
| 定位 |      基于'从零构建'的理念，用实战降低学习大模型的门槛，帮助程序员快速入门大模型。      |
| 愿景 | 若我的经验能让你的AI学习之路走的更容易一点，让你的生活变得更美好，我将倍感荣幸！祝好😄 |
+------+----------------------------------------------------------------------------------------+

环境信息：
+-------------+--------------+------------------------+
| Python 版本 | PyTorch 版本 | PyTorch Lightning 版本 |
+-------------+--------------+------------------------+
|   3.12.12   |    2.9.1     |         2.6.0          |
+-------------+--------------+------------------------+


## 准备数据

小美做了一个完美的示范：用 `2` 片柠檬片和 `16` 克糖，做了一杯好喝的柠檬水。

In [12]:
from course_01.util import print_table

# 准备训练数据
lemon = 2
sugar = 16

print_table(
    f"训练数据",
    ["柠檬片（输入数据）", "放糖量（结果标签）"],
    [[lemon, sugar]]
)


训练数据：
+--------------------+--------------------+
| 柠檬片（输入数据） | 放糖量（结果标签） |
+--------------------+--------------------+
|         2          |         16         |
+--------------------+--------------------+


## 模型定义

你为机器人创建了一个名为 `Model` 的模型，这个模型只有一个参数 `sugar_weight`，用于表示每片柠檬片所对应的放糖量，我们把它称为 `糖权`。

另外，我们：

- 把模型根据输入的柠檬片数量 `lemon` 预测放糖量、试做柠檬水的过程称为 `前向传播`，定义为 `forward` 方法
- 把与小美做的柠檬水进行比较的过程叫做 `损失计算`，定义为 `loss` 方法，它们之间的误差损失称为 `损失`
- 把根据 `损失` 判断增加或减少 `糖权` 的过程称为 `反向传播`，定义为 `backward` 方法  
- 把每一步具体更新 `糖权` 的过程称为 `参数更新`，定义为 `step` 方法

In [13]:
import random


class Model:

    def __init__(self):
        # self.sugar_weight = random.uniform(0, 10)
        # ❗️此处为了演示，将初始值固定为1.88，避免每次运行结果不一致
        self.sugar_weight = 1.88
        print(f"🚀 随机初始化的糖权: {self.sugar_weight}")

    # 1️⃣ 前向传播，计算预测值
    def forward(self, lemon):
        return self.sugar_weight * lemon

    # 2️⃣ 计算误差损失
    def loss(self, pred_sugar, real_sugar):
        return pred_sugar - real_sugar

    # 3️⃣ 计算更新参数的方向和步长
    def backward(self, loss):
        if loss > 0:
            # 如果损失值大于0，表示预测值偏大
            return -0.1
        elif loss < 0:
            # 如果损失值小于0，表示预测值偏小
            return 0.1

    # 4️⃣ 更新糖权
    def step(self, step):
        self.sugar_weight += step

## 模型训练与改进

### 模型训练

创建一个模型实例，并开始训练。

**训练的循环过程分为四步：**
1. 调用 `forward` 计算预测值
2. 调用 `loss` 计算预测值和真实值之间的误差
3. 调用 `backward` 根据误差判断参数更新方向
4. 调用 `step` 更新参数

每完成一次训练循环，我们称之为一个轮次，也就是 `epoch`。

下面我们对模型进行 `100` 个轮次的训练，查看模型训练学习的效果。

In [14]:
# 训练模型

# 创建模型实例
model = Model()

training_logs = []

# 训练模型
for epoch in range(100):
    # 1. 计算预测值
    pred = model.forward(lemon)
    # 2. 计算损失
    loss = model.loss(pred, sugar)
    # 3. 计算更新的梯度（方向）
    step = model.backward(loss)
    # 4. 更新参数
    model.step(step)

    # 记录训练日志，包括更新之后的权重（保留四位小数）
    # 按照表头顺序调整：Epoch, 柠檬盘, 当前糖权, 预测放糖量, 目标放糖量, 误差损失, 更新步长, 更新后权重
    if epoch < 5 or epoch > 95 or epoch % 5 == 0:
        training_logs.append([
            epoch,
            lemon,
            round(model.sugar_weight - step, 4),
            round(pred, 4),
            sugar,
            round(loss, 4),
            round(step, 4),
            round(model.sugar_weight, 4)
        ])

# 打印最终参数
print(f"✅ 模型学习到的糖权: {round(model.sugar_weight, 4)}")

print_table("📝 训练日志",
            ["Epoch", "柠檬片", "当前的糖权", "预测放糖量", "目标放糖量", "误差损失", "更新步长", "更新后糖权"],
            training_logs)

🚀 随机初始化的糖权: 1.88
✅ 模型学习到的糖权: 8.08

📝 训练日志：
+-------+--------+------------+------------+------------+----------+----------+------------+
| Epoch | 柠檬片 | 当前的糖权 | 预测放糖量 | 目标放糖量 | 误差损失 | 更新步长 | 更新后糖权 |
+-------+--------+------------+------------+------------+----------+----------+------------+
|   0   |   2    |    1.88    |    3.76    |     16     |  -12.24  |   0.1    |    1.98    |
|   1   |   2    |    1.98    |    3.96    |     16     |  -12.04  |   0.1    |    2.08    |
|   2   |   2    |    2.08    |    4.16    |     16     |  -11.84  |   0.1    |    2.18    |
|   3   |   2    |    2.18    |    4.36    |     16     |  -11.64  |   0.1    |    2.28    |
|   4   |   2    |    2.28    |    4.56    |     16     |  -11.44  |   0.1    |    2.38    |
|   5   |   2    |    2.38    |    4.76    |     16     |  -11.24  |   0.1    |    2.48    |
|   10  |   2    |    2.88    |    5.76    |     16     |  -10.24  |   0.1    |    2.98    |
|   15  |   2    |    3.38    |    6.76    |     16     | 

假设我们的机器人猜测的糖权是 `1.88` 克，每个训练轮次只让机器人更新 `0.1` 克的糖权。从小美的示范中我们可以看到，符合当地口味的最佳糖权是 `8.0`，这一点机器人是不知道的，机器人的目标就是找到这个隐藏的糖权。

我们来看一下具体的过程：
- 机器人做的第 1 杯柠檬水，使用的糖权是 `1.88` 克，放了 `3.76` 克的糖
- 经过和小美的示范对比后，发现太酸了，少了 `12.24` 克的糖，需要加糖
- 机器人将糖权调大了 `0.1` 克，现在的糖权从 `1.88` 克变成了 `1.98` 克
- 机器人做了第 2 杯柠檬水，使用的糖权是 `1.98` 克，放了 `3.96` 克的糖
- 经过和小美的示范对比后，发现还是太酸了，少了 `12.04` 克的糖，需要继续加糖
- 经过 `60` 次的反复尝试，机器人的糖权是 `7.98`，接近小美所使用的 `8.0` 克。

我们好像什么都没有做，但机器人却从一次一次的试错中真的逐渐学到了小美制作柠檬水的秘诀， magic！

### 训练改进

问题似乎解决了，但聪明的你发现了两个小漏洞：
- 首先，在接近最佳口味时，会错过最佳的糖权，导致要么太甜了一点，要么太酸了一点，机器人总是学习不到最佳的糖权。
- 其次，机器人每次制作柠檬水，都会消耗一定的成本。机器人经过几十上百次的尝试，消耗大量的成本才能学到小美的秘诀。

那么，怎么让机器人学的又快又好呢？

聪明的你马上又想到，我们能不能不让机器人每次都更新固定的糖权，而是让机器人从损失中动态的调整学习的幅度呢？

这里就要引入一个非常重要的概念：`学习率`。学习率决定了我们每次调整糖权的幅度，我们按照这个比例来调整糖权，差距大就调的多一点，差距小就调的少一点。这样既能保证学习效率，又不会因为调整幅度过大而错过最优解。

好想法！快来试试吧！

#### 模型的改进：引入学习率

在 `step` 方法中引入学习率参数，使得模型在训练过程中能够动态优化、快速收敛。

In [15]:
import random


class Model:

    def __init__(self):
        # 随机初始化糖权
        # self.sugar_weight = random.uniform(0, 50)
        self.sugar_weight = 1.88
        print(f"🚀 随机初始化的糖权: {self.sugar_weight}")

    # 1️⃣ 前向传播，计算预测值
    def forward(self, lemon):
        return self.sugar_weight * lemon

    # 2️⃣ 计算误差损失
    def loss(self, pred_sugar, real_sugar):
        return pred_sugar - real_sugar

    # 3️⃣ 🌟改进点 1 🌟：直接返回损失的值和更新的方向
    def backward(self, loss):
        # 使用负号控制更新的方向：太甜了就减糖，不够甜就加糖
        return -loss

    # 4️⃣ 🌟改进点 2 🌟：引入学习率动态的从损失中进行学习
    def step(self, loss, learning_rate):
        self.sugar_weight += loss * learning_rate

#### 学习的改进：使用学习率动态学习

将学习率设置为`10%`，更新参数时动态更新步长的十分之一，而不是固定更新`0.1`。

In [16]:
# 训练模型

# 创建模型实例
model = Model()

training_logs = []

learning_rate = 0.1

# 训练模型
for epoch in range(100):
    # 1. 计算预测值
    pred = model.forward(lemon)
    # 2. 计算损失
    loss = model.loss(pred, sugar)
    # 3. 计算更新的梯度（方向）
    step = model.backward(loss)
    # 4. 🌟改进点 🌟：引入学习率动态的从损失中进行学习更新参数
    model.step(step, learning_rate)

    # 记录训练日志，包括更新之后的权重（保留四位小数）
    # 按照表头顺序调整：Epoch, 柠檬盘, 当前糖权, 预测放糖量, 目标放糖量, 误差损失, 更新步长, 更新后权重
    if epoch < 5 or epoch > 95 or epoch % 5 == 0:
        training_logs.append([
            epoch,
            lemon,
            round(model.sugar_weight - step, 4),
            round(pred, 4),
            sugar,
            round(loss, 4),
            round(step, 4),
            round(model.sugar_weight, 4)
        ])

# 打印最终参数
print(f"✅ 模型学习到的糖权: {round(model.sugar_weight, 4)}")

print_table("📝 训练日志",
            ["Epoch", "柠檬片", "当前的糖权", "预测放糖量", "目标放糖量", "误差损失", "更新步长", "更新后糖权"],
            training_logs)

🚀 随机初始化的糖权: 1.88
✅ 模型学习到的糖权: 8.0

📝 训练日志：
+-------+--------+------------+------------+------------+----------+----------+------------+
| Epoch | 柠檬片 | 当前的糖权 | 预测放糖量 | 目标放糖量 | 误差损失 | 更新步长 | 更新后糖权 |
+-------+--------+------------+------------+------------+----------+----------+------------+
|   0   |   2    |   -9.136   |    3.76    |     16     |  -12.24  |  12.24   |   3.104    |
|   1   |   2    |  -5.7088   |   6.208    |     16     |  -9.792  |  9.792   |   4.0832   |
|   2   |   2    |   -2.967   |   8.1664   |     16     | -7.8336  |  7.8336  |   4.8666   |
|   3   |   2    |  -0.7736   |   9.7331   |     16     | -6.2669  |  6.2669  |   5.4932   |
|   4   |   2    |   0.9811   |  10.9865   |     16     | -5.0135  |  5.0135  |   5.9946   |
|   5   |   2    |   2.3849   |  11.9892   |     16     | -4.0108  |  4.0108  |   6.3957   |
|   10  |   2    |    6.16    |  14.6857   |     16     | -1.3143  |  1.3143  |   7.4743   |
|   15  |   2    |   7.3971   |  15.5693   |     16     | -

同样，我们的机器人猜测的糖权是 `1.88` 克，每次让机器人从差距中学习 `0.1` 比例来更新糖权。

我们来看一下具体过程：
- 机器人做的第一杯柠檬水，使用的糖权是 `1.88` 克，放了 `3.76` 克的糖
- 经过“撒糖哥”评估后，表示太酸了，少了 `12.24` 克的糖，需要加糖
- 机器人学习 `0.1` 比例的差距，将糖权调大了 `1.224` 克，现在的糖权是 `3.104` 克
- 这次只经过 `20` 次的尝试，机器人的糖权就达到了 `7.9436`，速度远远快于前面的学习方式

在第 `55` 次迭代时，机器人找到了最佳的糖权，这次没有在最佳位置来回的震荡。

太厉害了！我们的机器人彻底学会了小美制作柠檬水的秘诀， very very magic！


## 模型预测

我们的机器人已经学会了制作好喝的柠檬水，现在就来试试吧。

In [17]:
from course_01.util import print_regression_results

# 使用预训练的模型进行预测未见过的样本
lemons = [3, 4, 5]
real_sugars = [24, 32, 40]
pred_sugars = [model.forward(lemon) for lemon in lemons]

print_regression_results(lemons, real_sugars, pred_sugars, "🆕 制作新的柠檬水",
                         ["柠檬片", "真实放糖量", "预测放糖量", "预测偏差", "预测精度"])


🆕 制作新的柠檬水：
+--------+------------+------------+----------+----------+
| 柠檬片 | 真实放糖量 | 预测放糖量 | 预测偏差 | 预测精度 |
+--------+------------+------------+----------+----------+
|   3    |     24     |    24.0    | -0.0000  | 100.00%  |
|   4    |     32     |    32.0    | -0.0000  | 100.00%  |
|   5    |     40     |    40.0    | -0.0000  | 100.00%  |
+--------+------------+------------+----------+----------+


我们发现，虽然小美只给我们示范了如何使用 `2` 片柠檬片做柠檬水，但机器人却可以神奇的预测出 `3`、`4`、`5` 片柠檬所放的糖量，真是青出于蓝而胜于蓝，居然能处理从来没见过的新情况，具有非常好的泛化性！

:::{admonition} 提示 
:class: note
**简言之，机器学习就是通过数据驱动的方式，让计算机从经验中“学习”，用学到的经验来解决未见过但类似的新问题。**

**机器学习的实现可以分成两步：训练和预测，分别对应着归纳和演绎。**
- 归纳是“学”，是从具体到一般的总结。
- 演绎是“用”，是从一般到具体的应用。
:::

## 模型保存与加载

我们的第一个机器人已经学会制作好喝的柠檬水，那么我们怎么批量复制机器人，进行大规模投放呢？不会需要每次都重新训练，浪费大量的成本吧！

当然不需要！既然机器人已经学会了最佳的糖权（sugar_weight），我们可以将这个学到的参数保存下来，就像给机器人制作一个"记忆芯片"。这样，当我们需要制造新的机器人时，只需要加载这个"记忆芯片"，新机器人就能立即拥有制作完美柠檬水的技能，无需再经历漫长的训练过程。

接下来我们将学习如何将训练好的模型参数保存到文件中，以及如何从文件中加载这些参数来创建新的模型实例。

### 定义模型保存和加载方法

In [ ]:
import pickle


def save_parameters(model: Model, file_path):
    # 保存模型参数
    with open(file_path, "wb") as f:
        pickle.dump(model.sugar_weight, f)
    print(f"模型参数保存到 {file_path}")


def load_parameters(model: Model, file_path):
    # 加载模型参数
    with open(file_path, "rb") as f:
        model.sugar_weight = pickle.load(f)
    print(f"从 {file_path} 加载模型参数")

### 保存模型参数

保存模型时，只是简单地将参数 `sugar_weight` 的值保存到文件中。

In [ ]:
# 保存模型参数
save_parameters(model, "model.pkl")

### 加载模型参数

In [ ]:
# 加载训练好的模型
model = Model()
load_parameters(model, "model.pkl")
print(f"✅ 加载学习到的糖权: {model.sugar_weight:.4f}")

### 使用预训练的模型进行预测


In [ ]:
# 使用预训练的模型进行预测未见过的样本
lemons = [3, 4, 5]
real_sugars = [24, 32, 40]
pred_sugars = [model.forward(lemon) for lemon in lemons]

print_regression_results(lemons, real_sugars, pred_sugars, "🆕制作新的柠檬水",
                         ["柠檬片", "真实放糖量", "预测放糖量", "预测偏差", "预测精度"])

我们可以看到，新的机器人模型也能预测出 `3`、`4`、`5` 片柠檬所放的糖量，和我们的训练的第一个模型一样，效果都十分好。

我们把这种加载已经训练好的模型参数的方法，称为 `预训练`。我们从网络上下载的各种各样的大模型，都是预训练好的模型，所谓的模型就是各种参数值的集合而已！

通过模型的保存和加载功能，我们可以：
- 避免重复训练，节省计算资源和时间
- 在不同环境中部署相同的模型
- 持久化模型的训练成果，方便后续使用和分享

## 本章小结

本章通过“训练机器人制作柠檬水”这一生动故事，直观地阐述了机器学习的基本原理与完整工作流程。我们围绕一个核心任务——根据柠檬片数量预测放糖量，构建了一个仅含“糖权”单一参数的线性回归模型。

本章不仅手动实现了模型定义、前向传播、损失计算、反向传播与参数更新这一完整的训练循环，还揭示并解决了初始采用固定更新步长所导致的收敛缓慢及在最优解附近震荡的问题。

其关键改进在于引入了“学习率”这一核心概念，使得模型能够根据误差动态调整学习幅度，从而实现了更快速、精确的收敛。最终，模型不仅完美学习了训练数据中的配比关系，更展现出了优秀的泛化能力，能够准确预测未见过的样本。

这个简单的案例清晰地表明，机器学习本质上是一个从具体数据中“归纳”规律，并将规律“演绎”应用于新问题的过程。下一章，我们将以此为基础，探索更复杂的模型与更丰富的任务。

:::{admonition} 本章所学内容 
:class: note
- 机器学习就是在寻找一个“大公式”，通过数据驱动的方式，让计算机从过去的经验中“学习”这个大公式，用学到的知识来解决未见过但类似的新问题。
- 机器学习的实现可以分成两步：训练和预测，分别对应着归纳和演绎。理解这两者的关系，有助于更好地掌握机器学习的基本原理和应用方法。
- 机器学习的方法和人类学习的过程有着异曲同工之妙，在“机器学习”的过程中有三个关键要素：假设、评价、优化。
- 一个完整的模型训练流程包括：前向传播（计算预测值）、损失计算（对比预测与真实值）、反向传播（确定参数更新方向）和参数更新（执行优化）。
- “学习率”是控制参数更新幅度的重要超参数，合理的动态调度策略可以帮助模型更快、更好地完成训练，后续会使用优化器自动优化参数更新幅度，从而实现更优的训练效果。
- 模型在训练数据上学到的规律可以很好地泛化至未见过的数据上，这是机器学习实用价值的关键体现。
:::